# Training

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/Tienkhoaa2908/ISAS2026-BLE-location-prediction-team-SQ3K.git"
REPO_NAME = "ISAS2026-BLE-location-prediction-team-SQ3K"

if Path('/content').exists():
    work = Path('/content') / REPO_NAME
    if not work.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(work)], check=True)
    os.chdir(work)
else:
    cwd = Path.cwd()
    work = cwd if (cwd / 'requirements.txt').exists() else cwd.parent
    os.chdir(work)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repository =', Path.cwd())


## Data construction

In [ ]:
from pathlib import Path
import os, subprocess, sys

archive = Path(os.environ.get('SQ3K_DATA_ARCHIVE', '/content/data.zip'))

if not archive.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('data.zip was not uploaded')
        archive = Path('/content') / next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            'Set SQ3K_DATA_ARCHIVE to the private data.zip path before running locally.'
        )

runtime = Path('runtime/Dataset')
subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--archive', str(archive), '--out', str(runtime)
] , check=True)
print('data_root =', runtime.resolve())


In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/verify_fingerprints.py',
    '--data-root', 'runtime/Dataset',
    '--out', 'runtime/rebuilt_train_fingerprints.csv'
] , check=True)

fp = pd.read_csv('runtime/Dataset/train_fingerprints.csv')
feature_cols = [c for c in fp.columns if c not in ['date', 'win_start', 'win_end', 'room']]
print('rows =', len(fp))
print('classes =', fp['room'].astype(str).nunique())
print('numeric_features =', len(feature_cols))
display(fp['room'].astype(str).value_counts().rename('support').to_frame())


## Repeated session-grouped validation

In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/session_grouped_validation.py',
    '--data-root', 'runtime/Dataset',
    '--outdir', 'runtime/session_grouped',
    '--trees', '1200'
] , check=True)

summary = pd.read_csv('runtime/session_grouped/confirmation_summary.csv')
by_rep = pd.read_csv('runtime/session_grouped/confirmation_by_rep.csv')
display(summary)
display(by_rep)


## Leave-one-day-out validation

In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/lodo_validation.py',
    '--data-root', 'runtime/Dataset',
    '--outdir', 'runtime/lodo',
    '--specialist-trees', '1200',
    '--force'
] , check=True)

agg = pd.read_csv('runtime/lodo/paper_lodo_aggregate.csv')
by_day = pd.read_csv('runtime/lodo/paper_lodo_by_day.csv')
display(agg)
display(by_day.pivot(index='held_out_day', columns='system', values='macro_f1'))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred = pd.read_csv('runtime/lodo/paper_lodo_predictions.csv')
classes = sorted(pred['true'].astype(str).unique())
system = 'main_centered'
y_true = pred['true'].astype(str)
y_pred = pred[system].astype(str)

report = (pd.DataFrame(classification_report(
    y_true, y_pred, labels=classes, target_names=classes,
    output_dict=True, zero_division=0
)).T.loc[classes, ['precision', 'recall', 'f1-score', 'support']])
display(report)

cm = confusion_matrix(y_true, y_pred, labels=classes, normalize='true')
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(classes)), classes, rotation=90)
ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Fixed-22-class LODO — main + centered smoothing')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 4))
plt.bar(report.index, report['f1-score'])
plt.xticks(rotation=90)
plt.ylim(0, 1)
plt.ylabel('F1')
plt.xlabel('Location class')
plt.tight_layout()
plt.show()
